# ParkCast SF — Citywide Block Catalog Builder

Extends the served block universe from the ~1,413 metered blocks to all
12,242 blockfaces in `master_blocks.parquet`. For inferred (non-metered)
blocks, builds a **per-cnn** baseline lookup via distance-weighted KNN over
the K=5 nearest metered blocks. Each inferred block gets baselines that
reflect what real metered blocks near it look like at each (hour, dow),
rather than collapsing every block in a neighborhood to the same value.

**Produces:**
- `app/models/blocks.parquet` — citywide catalog with a `coverage` column (`metered` / `inferred`)
- `app/models/inferred_block_aggs.parquet` — KNN-derived per-(cnn, hour, dow) baselines for inferred blocks

The existing per-block lookup files (`block_aggs`, `block_static_features`,
`lag_history`) stay metered-only; the API code falls back to
`inferred_block_aggs` for blocks not present there.

## Imports

In [1]:
import os

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

## Configuration

In [2]:
PROJECT_DIR = os.path.dirname(os.getcwd())  # project root (run notebook from dev/)
MODEL_DIR = os.path.join(PROJECT_DIR, "app", "models")

MASTER_PATH = os.path.join(MODEL_DIR, "master_blocks.parquet")
METERED_BLOCKS_PATH = os.path.join(MODEL_DIR, "blocks.parquet")
BLOCK_AGGS_PATH = os.path.join(MODEL_DIR, "LightGBM.block_aggs.parquet")
OUT_BLOCKS_PATH = os.path.join(MODEL_DIR, "blocks.parquet")
OUT_INFERRED_AGGS_PATH = os.path.join(MODEL_DIR, "inferred_block_aggs.parquet")

# KNN parameters
KNN_K = 5
KNN_MAX_DIST_M = 1500     # don't reach across the city
KNN_DIST_FLOOR_M = 5.0    # min distance to avoid div-by-0 weights
DEG_TO_M = 111_000        # rough at SF latitude

# Reasonable default capacity when parking_census has no count (e.g. no_parking
# blocks, alleys). Median residential SF blockface is ~10 spaces.
DEFAULT_TOTAL_SPACES = 10

## Pipeline

### 1. Load source catalogs

Read the metered catalog from a `.bak` written by the deploy script before
running this; falls back to the current `blocks.parquet` on first run.

In [3]:
print("=" * 60)
print("ParkCast SF — Citywide block catalog builder")
print("=" * 60)

master = pd.read_parquet(MASTER_PATH)
src = METERED_BLOCKS_PATH + ".bak"
if not os.path.exists(src):
    src = METERED_BLOCKS_PATH
metered = pd.read_parquet(src)
aggs = pd.read_parquet(BLOCK_AGGS_PATH)
print(f"  master_blocks  : {len(master):,}")
print(f"  metered (src)  : {len(metered):,}  ({src})")
print(f"  block_aggs     : {len(aggs):,}")

ParkCast SF — Citywide block catalog builder
  master_blocks  : 12,242
  metered (src)  : 1,413  (/Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/app/models/blocks.parquet.bak)
  block_aggs     : 105,084


### 2. Citywide blocks catalog

Preserve the original metered catalog as-is (its lat/lon are the join keys
for `block_aggs` / `block_static_features` / `lag_history`). Add the
non-metered master blocks on top as inferred-coverage rows. Fill missing
`total_spaces` with the neighborhood median, falling back to a sensible
default for blocks the parking census didn't cover.

In [4]:
metered_keyset = set(zip(metered["lat"].round(6), metered["lon"].round(6)))

inferred = master[~master["is_metered"]].copy()
inferred["coverage"] = "inferred"

nbh_cap_med = (master.dropna(subset=["total_spaces"])
                     .groupby("neighborhood")["total_spaces"]
                     .median())
inferred["total_spaces"] = inferred.apply(
    lambda r: r["total_spaces"] if pd.notna(r["total_spaces"])
    else nbh_cap_med.get(r["neighborhood"], DEFAULT_TOTAL_SPACES),
    axis=1,
).round().clip(lower=1).astype(int)

inferred = inferred[[
    "lat", "lon", "neighborhood", "total_spaces", "coverage",
    "cnn", "block_class", "rpp_area",
]].dropna(subset=["lat", "lon", "neighborhood"])

# Don't shadow a metered row with a master midpoint that happens to land on
# the same coordinates (rare but possible).
inferred = inferred[~inferred.apply(
    lambda r: (round(r["lat"], 6), round(r["lon"], 6)) in metered_keyset, axis=1
)]

metered_out = metered.copy()
metered_out["coverage"] = "metered"

# Spatial-join metered blocks to master_blocks (nearest cnn) so each metered
# row gets cnn / block_class / rpp_area populated. Without this, downstream
# joins (street label, citation features, sensor truth, inferred_block_aggs)
# silently miss for metered blocks. Take the unconditional nearest — every
# metered block deserves a cnn even if the centerline midpoint is far from
# the meter centroid (long downtown blocks are the worst offenders).
master_keyed = master.dropna(subset=["cnn", "lat", "lon"]).copy()
master_keyed["cnn"] = master_keyed["cnn"].astype(int)
mtree = cKDTree(master_keyed[["lat", "lon"]].values)
_, midx = mtree.query(metered_out[["lat", "lon"]].values, k=1)
for col in ("cnn", "block_class", "rpp_area"):
    metered_out[col] = master_keyed[col].values[midx]
print(f"  metered rows joined to master cnn: {len(metered_out):,} / {len(metered_out):,} "
      f"({metered_out['cnn'].notna().mean()*100:.1f}%)")

metered_out = metered_out[[
    "lat", "lon", "neighborhood", "total_spaces", "coverage",
    "cnn", "block_class", "rpp_area",
]]

blocks = pd.concat([metered_out, inferred], ignore_index=True)

dups = blocks.duplicated(["lat", "lon"]).sum()
if dups:
    print(f"  WARN: {dups} duplicate (lat,lon) rows — deduplicating, keeping first")
    blocks = blocks.drop_duplicates(["lat", "lon"]).reset_index(drop=True)

blocks.to_parquet(OUT_BLOCKS_PATH, index=False)
print(f"  → {OUT_BLOCKS_PATH}  ({len(blocks):,} blocks)")
print(f"  cnn populated: {blocks['cnn'].notna().sum():,} / {len(blocks):,} "
      f"({blocks['cnn'].notna().mean()*100:.1f}%)")

  metered rows joined to master cnn: 1,413 / 1,413 (100.0%)
  → /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/app/models/blocks.parquet  (12,724 blocks)
  cnn populated: 12,724 / 12,724 (100.0%)


### 3. Per-cnn KNN baselines for inferred blocks

For each inferred (non-metered) block, find its K nearest metered blocks
(capped at 1.5km) and compute an inverse-distance-weighted average of their
per-(hour, dow) means. This gives every inferred block a baseline that
reflects the local metered behavior — a residential block 3 blocks south of
Irving St gets baselined by the closest few metered Irving blocks, not by
the entire Outer Sunset average.

In [5]:
metered_xy = blocks[blocks["coverage"] == "metered"][["lat", "lon"]].reset_index(drop=True)
inferred_xy = blocks[blocks["coverage"] == "inferred"][["cnn", "lat", "lon"]].reset_index(drop=True)
inferred_xy["cnn"] = inferred_xy["cnn"].astype("Int64")

tree = cKDTree(metered_xy[["lat", "lon"]].values)
dists_deg, idxs = tree.query(inferred_xy[["lat", "lon"]].values, k=KNN_K)
dists_m = dists_deg * DEG_TO_M

weights = 1.0 / np.maximum(dists_m, KNN_DIST_FLOOR_M)
weights[dists_m > KNN_MAX_DIST_M] = 0.0
print(f"  KNN built (K={KNN_K}, max_dist={KNN_MAX_DIST_M}m)")
print(f"  inferred blocks with at least one in-range neighbor: "
      f"{(weights.sum(axis=1) > 0).sum():,} / {len(inferred_xy):,}")

  KNN built (K=5, max_dist=1500m)
  inferred blocks with at least one in-range neighbor: 10,912 / 11,311


In [6]:
# Pivot block_aggs into a (n_metered, n_hour_dow) matrix for vectorized averaging.
hours_dows = (aggs[["hour", "day_of_week"]].drop_duplicates()
                  .sort_values(["hour", "day_of_week"]).reset_index(drop=True))
n_hd = len(hours_dows)
hd_to_col = {hd: i for i, hd in enumerate(zip(hours_dows["hour"], hours_dows["day_of_week"]))}

mlut = metered_xy.copy(); mlut["row"] = np.arange(len(metered_xy))
piv = aggs.merge(mlut, on=["lat", "lon"], how="inner")
piv["col"] = list(zip(piv["hour"], piv["day_of_week"]))
piv["col"] = piv["col"].map(hd_to_col)

def to_matrix(values):
    m = np.full((len(metered_xy), n_hd), np.nan, dtype=float)
    m[piv["row"].values, piv["col"].values] = values
    return m

M_bhd = to_matrix(piv["block_hour_dow_mean"].values)
M_bh  = to_matrix(piv["block_hour_mean"].values)
M_bm  = to_matrix(piv["block_mean"].values)
print(f"  metered × (hour,dow) matrix coverage: {(~np.isnan(M_bhd)).mean()*100:.1f}%")

  metered × (hour,dow) matrix coverage: 44.3%


In [7]:
def weighted_avg(matrix, w, neighbor_idxs):
    """For each inferred block (axis 0) and each hd-cell (axis 2),
    inverse-distance-weighted average of the K nearest metered values,
    skipping NaN cells."""
    vals = matrix[neighbor_idxs]              # (n_inf, K, n_hd)
    valid = ~np.isnan(vals)
    w3 = w[:, :, None] * valid
    wsum = w3.sum(axis=1)
    vsum = (np.where(valid, vals, 0.0) * w[:, :, None]).sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(wsum > 0, vsum / wsum, np.nan)

bhd = weighted_avg(M_bhd, weights, idxs)
bh  = weighted_avg(M_bh,  weights, idxs)
bm  = weighted_avg(M_bm,  weights, idxs)

n_inf = len(inferred_xy)
out = pd.DataFrame({
    "cnn": np.repeat(inferred_xy["cnn"].values, n_hd),
    "hour": np.tile(hours_dows["hour"].values, n_inf),
    "day_of_week": np.tile(hours_dows["day_of_week"].values, n_inf),
    "block_hour_dow_mean": bhd.flatten(),
    "block_hour_mean":     bh.flatten(),
    "block_mean":          bm.flatten(),
}).dropna(subset=["block_hour_dow_mean"]).reset_index(drop=True)

out.to_parquet(OUT_INFERRED_AGGS_PATH, index=False)
print(f"  → {OUT_INFERRED_AGGS_PATH}  ({len(out):,} rows, {out['cnn'].nunique():,} cnns)")

  → /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/app/models/inferred_block_aggs.parquet  (853,008 rows, 10,912 cnns)


### 4. Coverage + sanity check

Sanity: USF area at Tue 2pm should now show per-block variation, not a
uniform neighborhood mean.

In [8]:
print("Coverage breakdown:")
print(blocks["coverage"].value_counts().to_string())
print()

usf_cnn = set(master[master["neighborhood"].str.contains("Lone Mountain", na=False)]
                  .dropna(subset=["cnn"])["cnn"].astype(int))
out["cnn"] = out["cnn"].astype("Int64")
usf_2pm = out[out["cnn"].isin(usf_cnn) & (out["hour"] == 14) & (out["day_of_week"] == 2)]
if len(usf_2pm):
    print(f"USF Tue 2pm baselines (n={len(usf_2pm)}):")
    print(f"  median: {usf_2pm['block_hour_dow_mean'].median():.1f}%")
    print(f"  range:  {usf_2pm['block_hour_dow_mean'].min():.1f}-{usf_2pm['block_hour_dow_mean'].max():.1f}%")
    print(f"  std:    {usf_2pm['block_hour_dow_mean'].std():.1f}")
print("=" * 60)

Coverage breakdown:
coverage
inferred    11311
metered      1413



USF Tue 2pm baselines (n=118):
  median: 22.9%
  range:  12.0-52.5%
  std:    7.1
